In [2]:
import numpy as np
import pandas as pd
from joblib import dump, load
import os
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error, 
    mean_squared_error,
    root_mean_squared_error, 
    mean_absolute_percentage_error,
    root_mean_squared_log_error,
    make_scorer
)
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor

### Data importation

In [2]:
df = pd.read_csv("../data/processed/bc_clean.csv")
df = df.drop(["Unnamed: 0"], axis=1)
df.head(3)

,latitude,longitude,price,property-beds,property-baths,Acreage,Property Tax,Square Footage,Missing Acreage,Missing Property Tax,...,heat_pump,overhead,space_heater,Property Type_Condo,Property Type_Condo/Townhome,Property Type_Duplex,Property Type_Manufactured Home,Property Type_MultiFamily,Property Type_Single Family,Property Type_Townhome
0,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,6995.0,4374.0,0,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,49.138904,-122.654191,1399999.0,6.0,4.0,0.04,2585.0,2404.0,0,0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,49.103726,-122.663125,399900.0,1.0,1.0,0.00,1474.0,632.0,1,0,...,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
X = df.drop(["price"], axis=1)
Y = df["price"]

#### train test split

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, shuffle=True
)

#### BayesSearchCV

In [33]:
ada_search_space = {
    "n_estimators" : Integer(20, 600),
    "learning_rate" : Categorical([0.5, 1, 5]),

    "estimator__max_depth": Integer(3, 15),
    "estimator__min_samples_split" : Integer(30, 100),
    "estimator__min_samples_leaf" : Integer(20, 100),
    "estimator__max_features": Categorical([None, "sqrt", "log2"])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

In [36]:
ada_search = BayesSearchCV(
    estimator = AdaBoostRegressor(
        estimator = DecisionTreeRegressor(random_state=42),
        random_state=42
    ),
    search_spaces=ada_search_space,
    scoring = scoring["neg_rmsle"],
    n_iter = 50,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [3]:
if os.path.isfile("../artifacts/ada_search.pkl"):
    print("The object already exists")
else : 
    ada_search.fit(X_train, Y_train)
    dump(ada_search, "../artifacts/ada_search.pkl")
    print("The object has been successfully saved")

The object already exists


In [50]:
ada_search = load("../artifacts/ada_search.pkl")
ada_search.best_params_

OrderedDict([('estimator__max_depth', 15),
             ('estimator__max_features', None),
             ('estimator__min_samples_leaf', 20),
             ('estimator__min_samples_split', 30),
             ('learning_rate', 0.5),
             ('n_estimators', 20)])

In [55]:
best_model_ada = ada_search.best_estimator_

y_test_pred = best_model_ada.predict(X_test)
y_train_pred = best_model_ada.predict(X_train)

In [56]:
print("R²:", r2_score(Y_test, y_test_pred))
print("R²:", r2_score(Y_train, y_train_pred))

print("MAE:", mean_absolute_error(Y_test, y_test_pred))
print("MAE:", mean_absolute_error(Y_train, y_train_pred))

print("RMSE:", root_mean_squared_error(Y_test, y_test_pred))
print("RMSE:", root_mean_squared_error(Y_train, y_train_pred))

print("MAPE:", mean_absolute_percentage_error(Y_test, y_test_pred))
print("MAPE:", mean_absolute_percentage_error(Y_train, y_train_pred))

print("RMSLE", root_mean_squared_log_error(Y_test, y_test_pred))
print("RMSLE", root_mean_squared_log_error(Y_train, y_train_pred))

R²: 0.7660034906224046
R²: 0.95164835657712
MAE: 306021.11612481816
MAE: 221682.05524659943
RMSE: 837270.5165707233
RMSE: 403358.33935229207
MAPE: 0.18650127324937807
MAPE: 0.1571330858450282
RMSLE 0.25058488531221634
RMSLE 0.2118184958008385


We can observe a significative gap between the scores. The model is overfitting a little bit, so we'll try to reduce the learning_rate, on order to make the prediction converge slowly to the real price, without aggressivity.

In [57]:
df_error = pd.DataFrame({
    "y_true": Y_test,
    "y_pred": y_test_pred
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_65396/3528320552.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_65396/3528320552.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.296973
(629900.0, 888480.0]       0.119501
(888480.0, 1300000.0]      0.151340
(1300000.0, 2000000.0]     0.161855
(2000000.0, 29998000.0]    0.202777
dtype: float64

#### This time, we'll transform the Y vector target to a log-Y target vector in order to reduce the errors for the smallest prices of our dataset.  

Indeed, by applying the logarithm to the `Price` feature, the model penalizes much more the errors for the smallest price values than the biggest ones.  
We do that in order to give more importance to the errors on the smallest prices, by removing the dominance of large prices. We change the "space" of the error space thanks to the form of the logartihm function that becomes flatter for large values.

In [10]:
Y_log = np.log1p(Y)

#### New train test split

In [11]:
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(
    X, Y_log, test_size=0.20, random_state=42, shuffle=True
)

#### Search n°2  
We reduce our search space to focus on some promising points that reduce overfitting

In [61]:
ada_search_space2 = {
    "n_estimators" : Integer(30, 300),
    "learning_rate" : Real(0.01, 0.5),

    "estimator__max_depth": Integer(3, 15),
    "estimator__min_samples_split" : Integer(30, 100),
    "estimator__min_samples_leaf" : Integer(20, 100),
    "estimator__max_features": Categorical([None, "sqrt", "log2"])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

ada_search2 = BayesSearchCV(
    estimator = AdaBoostRegressor(
        estimator = DecisionTreeRegressor(random_state=42),
        random_state=42
    ),
    search_spaces=ada_search_space2,
    scoring = scoring["neg_rmsle"],
    n_iter = 60,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [4]:
if os.path.isfile("../artifacts/ada_search2.pkl"):
    print("The object already exists")
else : 
    ada_search2.fit(X_train2, Y_train2)
    dump(ada_search2, "../artifacts/ada_search2.pkl")
    print("The object has been successfully saved")

The object already exists


In [9]:
ada_search2 = load("../artifacts/ada_search2.pkl")
ada_search2.best_params_

OrderedDict([('estimator__max_depth', 15),
             ('estimator__max_features', None),
             ('estimator__min_samples_leaf', 20),
             ('estimator__min_samples_split', 30),
             ('learning_rate', 0.25871728802511557),
             ('n_estimators', 300)])

In [71]:
best_model_ada2 = ada_search2.best_estimator_

y_pred_log = best_model_ada2.predict(X_test2)
y_test_pred2 = np.expm1(y_pred_log)

y_train_pred_log = best_model_ada2.predict(X_train2)
y_train_pred2 = np.expm1(y_train_pred_log)

In [78]:
print("R²:", r2_score(np.expm1(Y_test2), y_test_pred2))
print("R²:", r2_score(np.expm1(Y_train2), y_train_pred2))

print("MAE:", mean_absolute_error(np.expm1(Y_test2), y_test_pred2))
print("MAE:", mean_absolute_error(np.expm1(Y_train2), y_train_pred2))

print("RMSE:", root_mean_squared_error(np.expm1(Y_test2), y_test_pred2))
print("RMSE:", root_mean_squared_error(np.expm1(Y_train2), y_train_pred2))

print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_test2), y_test_pred2))
print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_train2), y_train_pred2))

print("RMSLE", root_mean_squared_log_error(np.expm1(Y_test2), y_test_pred2))
print("RMSLE", root_mean_squared_log_error(np.expm1(Y_train2), y_train_pred2))

R²: 0.8004816615724227
R²: 0.9465568293523307
MAE: 289164.86047813826
MAE: 206421.53954168397
RMSE: 773129.9873957926
RMSE: 424064.1198642067
MAPE: 0.16280236080144975
MAPE: 0.11911561720071444
RMSLE 0.22254938231430363
RMSLE 0.1431156332991929


In [79]:
df_error = pd.DataFrame({
    "y_true": np.expm1(Y_test2),
    "y_pred": y_test_pred2
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_65396/3617340470.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_65396/3617340470.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(49499.999, 629900.0]      0.206155
(629900.0, 888480.0]       0.116646
(888480.0, 1300000.0]      0.138305
(1300000.0, 2000000.0]     0.158425
(2000000.0, 29998000.0]    0.194551
dtype: float64

#### Conclusion

AdaBoostRegressor with a DecisionTreeRegressor performs better than a Random Forest, without overfiting too.  
The mean average percentage error for each range of prices is less or equal to 20%, which is pretty good for a housing price estimator trained on a dataset with a large range of prices (from **38'800\\$** to **58'000'000\\$**).  
Even if the model gives us pretty realistic estimations, we'll train a GradientBoostingRegressor combined to a DecisionTreeRegressor, hoping to take advantage of its robustness and obtain better predictions.

In [10]:
if os.path.isfile("../artifacts/ada_best_model.pkl"):
    print("The object already exists")
else : 
    dump(ada_search2.best_estimator_, "../artifacts/ada_best_model.pkl")
    print("The object has been successfully saved")

The object has been successfully saved
